In [1]:
import pandas as pd
import json
import csv
import numpy as np

In [ ]:
# ========================================================================= #
# STEP ONE: Converting files
# ========================================================================= #

In [ ]:
# ========================================================================= #
# NASA CCMC DONKI
# .json -> .csv
# ========================================================================= #

In [2]:
# -------------------------------------------------------------------------------------- #
# CMEAnalysis.json (Coronal Mass Ejection) to .csv
# -------------------------------------------------------------------------------------- #
with open("CMEAnalysis.json") as f:
    data = json.load(f)
cme_df = pd.json_normalize(data)

# Convert time to a standardized time format
cme_df["time"] = pd.to_datetime(cme_df["time21_5"])

# Export
cme_df.to_csv("cme.csv", index=False)

# -------------------------------------------------------------------------------------- #
# FLR.json (Solar Flare) to .csv
# -------------------------------------------------------------------------------------- #
with open("FLR.json") as f:
    data = json.load(f)

# Main flare data
flr_df = pd.json_normalize(data)

flr_df["beginTime"] = pd.to_datetime(flr_df["beginTime"])
flr_df["peakTime"] = pd.to_datetime(flr_df["peakTime"])
flr_df["endTime"] = pd.to_datetime(flr_df["endTime"])

flr_df.to_csv("flr_main.csv", index=False)

# Instruments table
inst_df = pd.json_normalize(
    data,
    record_path="instruments",
    meta=["flrID"]
)
inst_df.to_csv("flr_instruments.csv", index=False)

# Linked events
linked_df = pd.json_normalize(
    data,
    record_path="linkedEvents",
    meta=["flrID"],
    errors="ignore"
)
linked_df.to_csv("flr_linked.csv", index=False)

# -------------------------------------------------------------------------------------- #
# GST.json (Geomagnetic Storms) to .csv
# -------------------------------------------------------------------------------------- #
with open("GST.json") as f:
    data = json.load(f)

gst_df = pd.json_normalize(data)
gst_df["startTime"] = pd.to_datetime(gst_df["startTime"])
gst_df.to_csv("gst_main.csv", index=False)

# Kp index values (Important for comparison later)
kp_df = pd.json_normalize(
    data,
    record_path="allKpIndex",
    meta=["gstID", "startTime"]
)
kp_df["observedTime"] = pd.to_datetime(kp_df["observedTime"])
kp_df.to_csv("kp_index.csv", index=False)

# -------------------------------------------------------------------------------------- #
# HSS.json (High Speed Streams) to .csv
# -------------------------------------------------------------------------------------- #
with open("HSS.json") as f:
    data = json.load(f)

hss_df = pd.json_normalize(data)
hss_df["eventTime"] = pd.to_datetime(hss_df["eventTime"])
hss_df.to_csv("hss_main.csv", index=False)

# Instruments
inst_df = pd.json_normalize(
    data,
    record_path="instruments",
    meta=["hssID"]
)
inst_df.to_csv("hss_instruments.csv", index=False)

# Linked events
linked_df = pd.json_normalize(
    data,
    record_path="linkedEvents",
    meta=["hssID"],
    errors="ignore"
)
linked_df.to_csv("hss_linked.csv", index=False)

In [ ]:
# ========================================================================= #
# NASA OMNI data 
# .lst -> .csv
# ========================================================================= #

In [3]:
# ============================================================
# Load multiple OMNI .lst files
# Convert missing values into NaN
# Build a proper datetime index
# Adds missingness indicators + gap sizes
# Interpolates ONLY short gaps
# Resamples to hourly resolution
# Exports to CSV
# ============================================================

# -------------------------------------------------------------------------------------- #
# Solar Wind Properties
# -------------------------------------------------------------------------------------- #
files = [
    "OMNI_5min_20100405_20140219.lst",
    "OMNI_5min_20140220_20180318.lst",
    "OMNI_5min_20180319_20220114.lst",
    "OMNI_5min_20220115_20260331.lst"
]

# ------------------------------------------------------------ #
# Column names based on OMNI format
# ------------------------------------------------------------ #
cols = [
    "year", "day", "hour", "minute",
    "B_total", "Bz", "speed",
    "density", "pressure", "E_field"
]

# Columns we want to specifically look at
feature_cols = ["B_total", "Bz", "speed", "density", "pressure", "E_field"]

# ------------------------------------------------------------ #
# Function to load and preprocess a single file
# ------------------------------------------------------------ #
MISSING_VALUES = {
    9999.99: np.nan,
    99999.9: np.nan,
    999.99: np.nan,
    99.99: np.nan
}

def load_omni_file(filepath):
    df = pd.read_csv(filepath, sep=r"\s+", names=cols)

    # Replace missing values
    df = df.replace(MISSING_VALUES)

    # Build datetime
    df["datetime"] = (
        pd.to_datetime(df["year"], format="%Y") +
        pd.to_timedelta(df["day"] - 1, unit="D") +
        pd.to_timedelta(df["hour"], unit="h") +
        pd.to_timedelta(df["minute"], unit="m")
    )

    df["datetime"] = df["datetime"].dt.tz_localize("UTC")

    return df

# ------------------------------------------------------------ #
# Load and combine all OMNI files
# ------------------------------------------------------------ #
dfs = [load_omni_file(f) for f in files]
omni = pd.concat(dfs, ignore_index=True)

omni = omni.sort_values("datetime").set_index("datetime")

# Regular, 5-minute grid
omni = omni.asfreq("5min")

# Missing value flags
for col in feature_cols:
    omni[f"{col}_is_missing"] = omni[col].isna().astype(int)

# Interpolation for gaps in missing data
MAX_GAP = 6  # 6 x 5min = 30 minutes

for col in feature_cols:
    omni[col] = omni[col].interpolate(
        method="time",
        limit=MAX_GAP,
        limit_direction="both"
    )
# ------------------------------------------------------------ #
# 5-Minute .csv file
# ------------------------------------------------------------ #
omni_5min_cleaned = omni.copy()
omni_5min_cleaned.to_csv(
    "omni_5min_cleaned.csv",
    date_format="%Y-%m-%d %H:%M:%S%z"
)

# ------------------------------------------------------------ #
# 1-Hour .csv file
# ------------------------------------------------------------ #
agg_dict = {}

# Continuous physical variables -> mean
for col in feature_cols:
    agg_dict[col] = "mean"

# Missing indicators -> preserve ANY missingness in hour (binary 0/1)
for col in feature_cols:
    agg_dict[f"{col}_is_missing"] = "max"

omni_1h = omni.resample("1h").agg(agg_dict)

# Hourly missing fraction feature (0.25 means 25% of 5-minute samples were missing)
for col in feature_cols:
    miss_col = f"{col}_is_missing"
    omni_1h[f"{col}_missing_fraction"] = (
        omni.resample("1h")[miss_col].mean()
    )

# Saving
omni_1h.to_csv(
    "omni_1hour_cleaned.csv",
    date_format="%Y-%m-%d %H:%M:%S%z"
)

# -------------------------------------------------------------------------------------- #
# Continuous kp index values (Kp*10 Index)
# -------------------------------------------------------------------------------------- #
# No missing values located

# Removing whitespaces
df = pd.read_csv("OMNI_1hr_Kp_20100405_20260331.lst", sep=r"\s+", header=None)

# Getting column names
df.columns = ["year", "doy", "hour", "Kp"]

# Standardizing the time
df["datetime"] = (
    pd.to_datetime(df["year"].astype(str) + df["doy"].astype(str), format="%Y%j")
    + pd.to_timedelta(df["hour"], unit="h")
).dt.tz_localize("UTC")

# Save to CSV
df.to_csv("kp_continuous.csv", date_format="%Y-%m-%d %H:%M:%S%z")

In [4]:
# ------------------------------------------------------------ #
# Sunspot Data Processing
# ------------------------------------------------------------ #
sun = pd.read_csv(
    "SILSO_SN_m_tot_V2.0.csv",
    sep=";",
    header=None
)

sun.columns = [
    "year", "month", "fractional_year",
    "sunspot_number", "std_dev",
    "observations", "provisional"
]

# Monthly sunspot data is assigned to the first day of each month.
sun["datetime"] = pd.to_datetime(
    sun["year"].astype(str) + "-" +
    sun["month"].astype(str) + "-01"
)

# Keep only the columns needed for merging into the master hourly dataset.
sun_monthly = sun[["datetime", "sunspot_number"]].copy()

In [ ]:
# ========================================================================= #
# STEP TWO: Merging & Standardizing time
# ========================================================================= #

In [5]:
# ------------------------------------------------------------ #
# Loading Data:
# ------------------------------------------------------------ #
# kp index from GST (Geomagnetic storms)
kp_gst = pd.read_csv("kp_index.csv")
# Continuous OMNI Kp history
kp_full = pd.read_csv("kp_continuous.csv")
# OMNIWeb (Solar wind properties)
omni = pd.read_csv("omni_1hour_cleaned.csv")
# Coronal mass ejections
cme = pd.read_csv("cme.csv")
# Solar flares
flr = pd.read_csv("flr_main.csv")
# High stream speeds
hss = pd.read_csv("hss_main.csv")

# ------------------------------------------------------------ #
# Standardizing Datetime more:
# (We already did this when converting, but just in case!)
# ------------------------------------------------------------ #
# GST storm-only Kp
kp_gst["datetime"] = pd.to_datetime(kp_gst["observedTime"]).dt.tz_localize(None)
# Renaming to differentiate Kp for comparison
kp_gst = kp_gst.rename(columns={"kpIndex": "Kp_gst"})

# Continuous Kp from OMNI
kp_full["datetime"] = pd.to_datetime(kp_full["datetime"]).dt.tz_localize(None)
# Convert Kp*10 -> real Kp scale
kp_full["Kp_omni"] = kp_full["Kp"] / 10.0

# OMNI wind
omni["datetime"] = pd.to_datetime(omni["datetime"]).dt.tz_localize(None)

# Coronal Mass Ejections
cme["datetime"] = pd.to_datetime(cme["time"]).dt.tz_localize(None)

# Solar flares (Using flare peak time) 
flr["datetime"] = pd.to_datetime(flr["peakTime"]).dt.tz_localize(None)

# High speed streams
hss["datetime"] = pd.to_datetime(hss["eventTime"]).dt.tz_localize(None)

# ------------------------------------------------------------ #
# Base hourly timeline
# ------------------------------------------------------------ #
start_time = max(
    kp_full["datetime"].min(),
    omni["datetime"].min()
)

end_time = min(
    kp_full["datetime"].max(),
    omni["datetime"].max()
)

master = pd.DataFrame({
    "datetime": pd.date_range(
        start=start_time,
        end=end_time,
        freq="1h"
    )
})

# ------------------------------------------------------------ #
# Merging files
# ------------------------------------------------------------ #
# Kp from OMNI
master = master.merge(
    kp_full[["datetime", "Kp_omni"]],
    on="datetime",
    how="left"
)

# Official GST Kp comparison values
master = master.merge(
    kp_gst[["datetime", "Kp_gst", "gstID", "startTime"]],
    on="datetime",
    how="left"
)

# OMNI physical variables
master = master.merge(
    omni,
    on="datetime",
    how="left"
)

# Sunspot data
# Sunspot data is monthly, while the master dataset is hourly
# Create a month-start column so each hourly row can inherit the corresponding monthly sunspot number
master["month_start"] = master["datetime"].dt.to_period("M").dt.to_timestamp()

master = master.merge(
    sun[["datetime", "sunspot_number"]],
    left_on="month_start",
    right_on="datetime",
    how="left",
    suffixes=("", "_sun")
)

master.drop(columns=["datetime_sun", "month_start"], inplace=True)

In [ ]:
# ========================================================================= #
# STEP THREE: Feature Engineering
# ========================================================================= #

In [ ]:
# Convert event times into an hourly time series and use rolling windows
# ------------------------------------------------------------ #
# CME features
# ------------------------------------------------------------ #
cme = cme.sort_values("datetime").copy()

# Count how many CME events occurred in each hour, then count the last 72 hours.
cme_hourly_count = (
    cme.set_index("datetime")
    .assign(cme_event=1)["cme_event"]
    .resample("1h")
    .sum()
)

master = master.merge(
    cme_hourly_count
    .rolling("72h", min_periods=1)
    .sum()
    .rename("num_cme_last_72h"),
    left_on="datetime",
    right_index=True,
    how="left"
)

# Maximum CME speed in the last 72 hours.
if "speed" in cme.columns:
    cme_hourly_speed = cme.set_index("datetime")["speed"].resample("1h").max()

    master = master.merge(
        cme_hourly_speed
        .rolling("72h", min_periods=1)
        .max()
        .rename("max_cme_speed_last_72h"),
        left_on="datetime",
        right_index=True,
        how="left"
    )
else:
    master["max_cme_speed_last_72h"] = 0

master["num_cme_last_72h"] = master["num_cme_last_72h"].fillna(0)
master["max_cme_speed_last_72h"] = master["max_cme_speed_last_72h"].fillna(0)

# ------------------------------------------------------------ #
# Flare features
# ------------------------------------------------------------ #
flr = flr.sort_values("datetime").copy()
flr["is_mx_flare"] = flr["classType"].astype(str).str.startswith(("M", "X")).astype(int)

flare_hourly_count = (
    flr.set_index("datetime")
    .assign(flare_event=1)["flare_event"]
    .resample("1h")
    .sum()
)

mx_flare_hourly_count = (
    flr.set_index("datetime")["is_mx_flare"]
    .resample("1h")
    .sum()
)

master = master.merge(
    flare_hourly_count
    .rolling("24h", min_periods=1)
    .sum()
    .rename("num_flares_last_24h"),
    left_on="datetime",
    right_index=True,
    how="left"
)

master = master.merge(
    mx_flare_hourly_count
    .rolling("24h", min_periods=1)
    .sum()
    .rename("num_mx_flares_last_24h"),
    left_on="datetime",
    right_index=True,
    how="left"
)

master["num_flares_last_24h"] = master["num_flares_last_24h"].fillna(0)
master["num_mx_flares_last_24h"] = master["num_mx_flares_last_24h"].fillna(0)

# ------------------------------------------------------------ #
# HSS features
# ------------------------------------------------------------ #
hss = hss.sort_values("datetime").copy()

hss_hourly_count = (
    hss.set_index("datetime")
    .assign(hss_event=1)["hss_event"]
    .resample("1h")
    .sum()
)

master = master.merge(
    hss_hourly_count
    .rolling("72h", min_periods=1)
    .sum()
    .rename("num_hss_last_72h"),
    left_on="datetime",
    right_index=True,
    how="left"
)

master["num_hss_last_72h"] = master["num_hss_last_72h"].fillna(0)
master["hss_active"] = (master["num_hss_last_72h"] > 0).astype(int)

# ------------------------------------------------------------ #
# Rolling features for OMNI
# ------------------------------------------------------------ #
# Bz
if "Bz" in master.columns:

    master["bz_mean_3h"] = master["Bz"].rolling(3).mean()
    master["bz_min_6h"] = master["Bz"].rolling(6).min()
    master["bz_hours_negative_12h"] = (
        (master["Bz"] < 0)
        .rolling(12)
        .sum()
    )

# Speed
if "speed" in master.columns:

    master["speed_mean_6h"] = master["speed"].rolling(6).mean()
    master["speed_change_3h"] = master["speed"] - master["speed"].shift(3)

# Pressure
if "pressure" in master.columns:

    master["pressure_max_6h"] = master["pressure"].rolling(6).max()

# Electric field
if "E_field" in master.columns:

    master["efield_mean_3h"] = master["E_field"].rolling(3).mean()

# ------------------------------------------------------------ #
# [!!!] Target Variables [!!!]
# ------------------------------------------------------------ #
# Predict max Kp next 6 hours
FORECAST_HORIZON = 6
STORM_THRESHOLD = 5.0

# Future max Kp from t+1 through t+6
master["kp_future_6h_max"] = (
    master["Kp_omni"]
    .shift(-1)
    .rolling(window=FORECAST_HORIZON, min_periods=FORECAST_HORIZON)
    .max()
    .shift(-(FORECAST_HORIZON - 1))
)

# Binary target: will a geomagnetic storm occur in the next 6 hours?
master["storm_next_6h"] = (
    master["kp_future_6h_max"] >= STORM_THRESHOLD
).astype(int)

# Current storm flag, useful for comparison but NOT used as input
master["storm_now"] = (
    master["Kp_omni"] >= STORM_THRESHOLD
).astype(int)

# ------------------------------------------------------------ #
# Storm persistence features
# ONLY uses past kp information
# ------------------------------------------------------------ #
master["storm_past_3h"] = (
    master["storm_now"]
    .shift(1)
    .rolling(3, min_periods=1)
    .sum()
)

master["storm_past_6h"] = (
    master["storm_now"]
    .shift(1)
    .rolling(6, min_periods=1)
    .sum()
)

master["kp_past_3h_max"] = (
    master["Kp_omni"]
    .shift(1)
    .rolling(3, min_periods=1)
    .max()
)

master["kp_past_6h_max"] = (
    master["Kp_omni"]
    .shift(1)
    .rolling(6, min_periods=1)
    .max()
)

# ------------------------------------------------------------ #
# [!!!] Hours until next actual storm [!!!]
# ------------------------------------------------------------ #
# Vectorized for efficiency
# This finds the next storm index for every row.
storm_now_indices = master.index[master["storm_now"] == 1].to_numpy()
row_positions = np.arange(len(master))

# searchsorted gives the location of the first storm index greater than each row.
next_storm_position = np.searchsorted(storm_now_indices, row_positions + 1)
valid_next_storm = next_storm_position < len(storm_now_indices)

master["hours_until_storm"] = np.nan
master.loc[valid_next_storm, "hours_until_storm"] = (
    storm_now_indices[next_storm_position[valid_next_storm]]
    - row_positions[valid_next_storm]
)

# Removes rows that don't have complete future labels
master = master.dropna(subset=["kp_future_6h_max"]).reset_index(drop=True)

# Export
master.to_csv(
    "master_geomagnetic_1hour.csv",
    index=False
)